## Operator Study

This notebook focuses on studying single operator surgery, comparing different approaches against baseline. We also test different compression strategies and the effects of combining them (operator replacement, low-rank factorization, ...)

### Operator architecture

For the current dense-model scope, write the teacher SwiGLU MLP at layer $l$ as

$$
f_l(h)=W_{\mathrm{down}}\left[\mathrm{SiLU}(W_{\mathrm{gate}}h)\odot(W_{\mathrm{up}}h)\right],
$$

$h$ is the normalized MLP input and $f_l(h)$ is the contribution returned to the residual stream. A drop-in operator $\hat f_l$ replaces only $f_l$, with the same input and output shape; the surrounding Transformer block remains unchanged.

### Operator classes: Design space relevant to this thesis

The table maps candidate operators to the question each one can answer. Parameter scales ignore biases, with $d=d_{\mathrm{model}}$ and reduced width or rank $r<d$.

| Operator class | Representative form | Approximate parameters | Experimental role | Baseline |
| --- | --- | ---: | --- | :---: |
| Zero / mean controls | $0$ or $\mu_y$ | 0 trainable | Bound complete removal and input-independent prediction. | true |
| Dense affine map | $Ax+b$ | $d^2+d$ | Test whether one learned affine transformation is sufficient. | true |
| Low-rank affine map | $U(Vx)+b$ | $2dr+d$ | Test whether linear structure is sufficient under a rank bottleneck. | false |
| Compact ungated MLP | $W_2\phi(W_1x)$ | $2dr$ | Isolate the value of nonlinearity without multiplicative gating. | false |
| Reduced-width SwiGLU | $W_d[\mathrm{SiLU}(W_gx)\odot(W_ux)]$ | $3dr$ | Preserve the teacher family at lower width; this is the current practical baseline. | true |
| Linear plus nonlinear correction | $U(Vx)+W_2\phi(W_1x)$ | $2d(r_{\mathrm{lin}}+r_{\mathrm{nl}})$ | Test whether most behavior is low-rank linear with a small nonlinear residual. | false |
| Factorized teacher projections | $W_j\approx U_jV_j$ inside SwiGLU | depends on selected matrices and ranks | Compress original weights while retaining the gate and activation structure. | false |
| Partial internal replacement | Replace only a projection, gate branch, or activation path | design-specific | Test whether targeted surgery outperforms replacing the complete MLP. | false |

Whole-block operators are studied first because they share one clean drop-in interface. Partial replacement and factorization are later experiments and require their own controls.

**Status and citation requirement.** This table is a project design-space synthesis, not a taxonomy copied from one paper. The affine, low-rank, compact-MLP, and hybrid equations are explanatory operator definitions and do not require citations as proposed experimental variants. Any claim that a particular published method uses or benefits from one of them must cite that method.

### Low-rank factorization: two distinct uses

1. **Whole-operator approximation:** $\hat f(x)=U(Vx)+b$ replaces the nonlinear MLP with one low-rank affine function.
2. **Internal weight factorization:** $W_j\approx U_jV_j$ replaces selected SwiGLU weight matrices while retaining the nonlinear gated computation.

These approaches may use the same matrix factorization machinery, but they test different hypotheses and should be reported separately.

### Experiment ideas

1. Operator class testing - fit and test full operators (dense) - test loss (local), loss (model-wide e.g. ppl)
2. Apply algebraic redundancy techniques - low-rank factorization

- we can first maybe test full operators, then maybe advanced operators and as chaining of compression techniques compare operators to factorization optimized operators to see differences and tradeoffs

- idea: plot multiple selected blocks statistics (e.g. layer=3, 10, 18) to see how it diverges/converges, analyzing block depth behaviour -> hue=layer_index


### Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlp_replacement.capture import collect_module_io
from mlp_replacement.config import DataConfig, ModelConfig, OperatorConfig
from mlp_replacement.data import build_data_loaders
from mlp_replacement.evaluation.language_model import evaluate_language_model
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import get_mlp_block, load_model_and_tokenizer
from mlp_replacement.operators import (
    BottleneckMLPReplacement,
    GatedMLPReplacement,
    HybridReplacement,
    LinearReplacement,
    LowRankLinearReplacement,
    MeanReplacement,
    ZeroReplacement,
    fit_operator,
    fit_ridge_linear,
    initialize_low_rank_from_linear,
    linear_svd,
)
from mlp_replacement.surgery import (
    count_parameters,
    count_state_elements,
    temporary_replacement,
)

In [ ]:
SEED = 21
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
TARGET_LAYER = 11
LINEAR_RIDGE = 1e-4

In [ ]:
model_config = ModelConfig(
    model_id='HuggingFaceTB/SmolLM2-1.7B',
    device='auto',
    dtype='auto',
)

data_config = DataConfig(
    sequence_length=128,
    batch_size=2,
    num_calibration_batches=48,
    num_operator_validation_batches=24,
    num_recovery_batches=0,
    num_recovery_validation_batches=0,
    num_model_validation_batches=24,
    num_test_batches=0,
    seed=SEED,
)

training_config = OperatorConfig(
    epochs=64,
    learning_rate=1e-3,
    batch_size=2048,
    weight_decay=0.0,
    scheduler='constant',
    early_stopping_patience=3,
    seed=SEED,
)

In [ ]:
model, tokenizer = load_model_and_tokenizer(model_config)
device = next(model.parameters()).device
block = get_mlp_block(model, TARGET_LAYER)

In [ ]:
loaders = build_data_loaders(
    tokenizer,
    data_config,
    include_recovery=False,
)

## Experiments

### Operator classes

#### swiGLU expansion ratio analysis

- testing range of [0.10, ..., 0.9]
- importance link

### Advanced Personalized Operators

### Model Integration Loss analysis

- comparing different operators and required re-training budgets for performance convergence
    - answers: how much compute does operator need to stabilize model loss (ppl)?

### Applied secondary compression techniques

1. low-rank factorization (algebraic redundancy)
2. Quantization (numeric redundancy) [optional]